In [7]:
import pandas as pd
import time
import os
import google.generativeai as genai
from tqdm import tqdm
# from google.colab import drive

/Users/justin/Documents/ITB/Semester 7/Natural Language Processing/Tugas Besar/Tugas-Besar-NLP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# --- CONFIGURATION ---
API_KEY = ""
MODEL_ID = "gemini-2.5-flash-lite"
START_ROW = 0
END_ROW = 999

# File paths
INPUT_FILENAME = 'sampled_tourist_reviews.csv'
OUTPUT_FILENAME = 'dataset_translated_0_999.csv'

genai.configure(api_key=API_KEY)

def translate_text(text):
    """
    Tries to translate text. 
    - Retries automatically on 503 (Overloaded) and 429 (Rate Limit).
    """
    max_retries = 5
    base_delay = 5

    for attempt in range(max_retries):
        try:
            model = genai.GenerativeModel(MODEL_ID)
            response = model.generate_content(
                f"Translate to English. Output ONLY the translation. Text: {text}"
            )
            return response.text.strip()

        except Exception as e:
            error_str = str(e)
            
            if "503" in error_str or "overloaded" in error_str:
                wait_time = base_delay * (2 ** attempt)
                print(f"\n  ⚠️ Server overloaded (503). Retrying in {wait_time}s...")
                time.sleep(wait_time)
                continue
            
            elif "429" in error_str or "RESOURCE_EXHAUSTED" in error_str:
                wait_time = 4  # Wait 4 seconds for rate limit
                print(f"\n  ⚠️ Rate limit hit (429). Waiting {wait_time}s before retry...")
                time.sleep(wait_time)
                continue
            
            else:
                print(f"\n  ❌ Unexpected Error: {e}")
                return None

    return None

def main():
    # Ensure input file exists
    if not os.path.exists(INPUT_FILENAME):
        print(f"❌ Error: Could not find input file: {INPUT_FILENAME}")
        return

    # Resume Logic
    if os.path.exists(OUTPUT_FILENAME):
        print(f"🔄 Resuming from existing file: {OUTPUT_FILENAME}")
        df = pd.read_csv(OUTPUT_FILENAME)
    else:
        print(f"🆕 Starting fresh. Reading: {INPUT_FILENAME}")
        df = pd.read_csv(INPUT_FILENAME)
        if 'english_translation' not in df.columns:
            df['english_translation'] = None

    # Filter rows that are still null
    # We convert to object type to avoid pandas warnings
    df['english_translation'] = df['english_translation'].astype(object)
    
    # Get all indices where translation is missing
    missing_rows = df[df['english_translation'].isnull()].index
    
    # Apply the START_ROW and END_ROW filter
    # This keeps only rows that are missing AND within the specified range
    rows_to_process = [
        idx for idx in missing_rows 
        if idx >= START_ROW and (END_ROW is None or idx < END_ROW)
    ]
    
    print(f"Total rows in file: {len(df)}")
    print(f"Rows missing translation: {len(missing_rows)}")
    print(f"Configuration: START_ROW={START_ROW}, END_ROW={END_ROW}")
    print(f"Actual rows to process: {len(rows_to_process)}")
    print(f"Saving every 100 rows...")
    print("------------------------------------------------")

    count = 0
    
    for idx in tqdm(rows_to_process):
        original_text = df.at[idx, 'text']
        
        if pd.isna(original_text) or str(original_text).strip() == "":
            df.at[idx, 'english_translation'] = ""
            continue

        translation = translate_text(original_text)
        
        if translation:
            df.at[idx, 'english_translation'] = translation
        
        count += 1

        # --- SAVE EVERY 100 ROWS ---
        if count % 100 == 0:
            df.to_csv(OUTPUT_FILENAME, index=False)
        
        # Rate Limit: 15 requests per minute = 4 seconds per request
        time.sleep(4)

    # Final Save
    df.to_csv(OUTPUT_FILENAME, index=False)
    print(f"✅ Process ended. Saved final file to: {OUTPUT_FILENAME}")

if __name__ == "__main__":
    main()

🆕 Starting fresh. Reading: sampled_tourist_reviews.csv
Total rows in file: 6000
Rows missing translation: 6000
Configuration: START_ROW=0, END_ROW=100
Actual rows to process: 100
Saving every 100 rows...
------------------------------------------------


  4%|▍         | 4/100 [01:48<45:08, 28.21s/it]


  ⚠️ Server overloaded (503). Retrying in 5s...


  6%|▌         | 6/100 [02:53<45:25, 29.00s/it]



⚠️ DAILY QUOTA REACHED (Error 429) ⚠️
Saving progress and stopping...
✅ Process ended. Saved final file to: dataset_translated.csv
